# 発展演習 — AIが自分でコードを直すループ
### モデルにコードを書かせ、ツールが検査し、失敗したらモデルに直させる

モジュール3では、あえて**1回だけ判断して終わる**エージェントを作りました。この
演習では、その先——**書く→検査する→直す**を繰り返す、少しだけ賢いループを
作ります。

**使うモデル:** モジュール3で使った、同じ`Qwen2.5-0.5B-Instruct`(S3経由・
ローカル実行、追加費用なし)。新しいAPIキーや外部サービスは一切必要ありません。

**流れ:**
1. モデルに、ある処理を行うPython関数を書いてもらう
2. 「ツール」(`ast.parse`という、Python標準の構文チェック機能)が、
   そのコードが文法的に正しいかを検査する
3. もし間違っていたら、**エラーメッセージをそのままモデルに見せて**、
   直してもらう
4. 正しくなるまで(または規定回数まで)繰り返す

> 💡 **このノートブックも2つのモードで動きます:**
> - `USE_MOCK_LLM = True`(デフォルト): あらかじめ用意した「わざと1回失敗する」
>   コード生成を再現し、ループの仕組み自体を確認します
> - `USE_MOCK_LLM = False`: 本物のモデルに実際にコードを書かせます。
>   モジュール3のボーナスと同じで、結果は毎回変わる可能性があります

## 環境セットアップ

- **インスタンスタイプ:** `ml.t3.medium`(本物のモデルを試すなら`ml.t3.large`)
- **カーネル:** `conda_pytorch_p310`

In [ ]:
# conda_pytorch_p310 にはPyTorchは入っていますが、transformersは
# 入っていない場合があるため、明示的にインストールします
# (このノートブックはMCPサーバーを使わないため、fastmcpは不要です)
%pip install --quiet transformers

In [ ]:
# セットアップ
import ast
import re

USE_MOCK_LLM = True  # 本物のモデルを試す準備ができたら False に変更してください
MODEL_SOURCE = "local"
LOCAL_MODEL_PATH = "./models/Qwen2.5-0.5B-Instruct"

_generator_cache = {}

def get_generator():
    """モジュール3と同じ、モデル読み込みの共通処理です。"""
    if MODEL_SOURCE in _generator_cache:
        return _generator_cache[MODEL_SOURCE]
    import os, warnings
    os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
    warnings.filterwarnings("ignore")
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
    from transformers import pipeline
    model_ref = LOCAL_MODEL_PATH if MODEL_SOURCE == "local" else "Qwen/Qwen2.5-0.5B-Instruct"
    generator = pipeline("text-generation", model=model_ref)
    generator.model.generation_config.max_length = None
    _generator_cache[MODEL_SOURCE] = generator
    return generator

print(f"準備完了です。USE_MOCK_LLM = {USE_MOCK_LLM}")

### 本物のモデルを使う予定がある場合は、ここで取得しておいてください(任意)

このノートブックの後半で`USE_MOCK_LLM = False`に切り替えると、本物の指示
モデル(`Qwen2.5-0.5B-Instruct`)が必要になります。**モデルはステップ1より
前に読み込まれる**ため、後半まで取得を待つと、途中のセルでエラーになります
(モジュール3で実際に確認された問題と同じです)。

本物のモデルを試す予定があるなら、今のうちに次のセルを実行しておいてください。
今日は`USE_MOCK_LLM = True`(デフォルト)のまま進めるなら、このセルは
**実行しなくてもかまいません**(スキップしても後の演習に影響しません)。

In [ ]:
# このワークショップ用に、モデル一式をS3バケット(公開読み取り可能)に事前配置しています。
# --no-sign-request を付けることで、AWS認証情報なしで公開バケットから取得できます。
!aws s3 sync s3://llm-workshop-files/Qwen2.5-0.5B-Instruct ./models/Qwen2.5-0.5B-Instruct --no-sign-request

## ステップ1: 「ツール」— 構文チェッカー

これが今日の「ツール」です。`ast.parse`はPython標準ライブラリの一部で、
コードを実際に実行せずに、**文法として正しいかどうか**だけを調べます。

これはモジュール2の`check_inventory`と同じ考え方の「ツール」です——
明確な入力(コード文字列)を受け取り、明確な出力(正しいか、エラー内容か)を
返します。ちなみに、これはAIでも「エキスパートシステム」でもありません——
Python自体が実行前に必ず行う、決定的な文法チェックそのものです。

In [ ]:
def check_code(code_str: str):
    """コード文字列がPythonとして構文的に正しいかを調べる。
    正しければ (True, None)、間違っていれば (False, エラーメッセージ) を返す。"""
    try:
        ast.parse(code_str)
        return True, None
    except SyntaxError as e:
        return False, f"{type(e).__name__}: {e}"

# 動作確認
print(check_code("def add(a, b):\n    return a + b"))       # 正しいコード
print(check_code("def add(a, b)\n    return a + b"))         # コロンが抜けている(わざと)

## ステップ2: モデルへの「指示文」を組み立てる

モデルに送る指示文(プロンプト)を、専用の関数として切り出します。こうする
ことで、**モデルに実際に何が送られているか**を、後で覗けるようにします。
2回目以降は、前回のエラーメッセージも指示文に含めます。

In [ ]:
def build_prompt(task: str, previous_error: str = None) -> str:
    """モデルに送る指示文を組み立てる。previous_errorがあれば、
    「前回はこのエラーで失敗した」という情報も指示文に含める。"""
    if previous_error is None:
        return (
            f"Write a single Python function that does the following: {task}\n"
            f"Return ONLY a python code block."
        )
    else:
        return (
            f"Your previous code for this task failed with this error:\n{previous_error}\n"
            f"Task: {task}\nFix the code. Return ONLY a corrected python code block."
        )

def extract_code(raw_text: str) -> str:
    """モデルの出力からPythonコードブロックだけを取り出す。
    コードブロックが見つからなければ、出力全体をそのまま返す。"""
    match = re.search(r"```(?:\w+)?\n(.*?)```", raw_text, re.DOTALL)
    return match.group(1).strip() if match else raw_text.strip()

def mock_generate_code(prompt: str) -> str:
    """模擬関数:指示文に前回のエラーが含まれていなければ、わざと構文エラーの
    あるコードを返します。含まれていれば、正しいコードを返します。"""
    if "failed with this error" not in prompt:
        return "```python\ndef is_even(n)\n    return n % 2 == 0\n```"
    else:
        return "```python\ndef is_even(n):\n    return n % 2 == 0\n```"

def real_generate_code(prompt: str) -> str:
    """本物のバージョン。USE_MOCK_LLM = False のときだけ使われます。"""
    generator = get_generator()
    messages = [{"role": "user", "content": prompt}]
    output = generator(messages, max_new_tokens=150)
    return output[0]["generated_text"][-1]["content"]

generate_code = mock_generate_code if USE_MOCK_LLM else real_generate_code
print("使用中:", generate_code.__name__)

## ステップ3: 書く→検査する→直す、のループ

**あなたの番:** `self_correcting_loop`を完成させてください:
1. `build_prompt(task, previous_error)`で指示文を作る
2. `generate_code(prompt)`でコードを生成する
3. `check_code(...)`で検査する
4. 正しければ、そのコードを返して終了
5. 間違っていれば、エラー内容を`previous_error`として次のループに渡し、
   `max_attempts`回まで繰り返す

> 📌 **`verbose=True`にすると、モデルとチェッカーの「やり取り」がすべて
> 画面に表示されます。** 何が起きているかを実際に覗いてみたい場合は
> `verbose=True`のまま、結果だけを見たい場合は`verbose=False`にして
> 実行してください。

In [ ]:
def TODO(hint=""):
    """演習の未完成部分を示す関数です。TODO(...)の呼び出しを自分のコードに
    置き換えてください。置き換えるまではエラーが出続けます(これは正常な動作です)。"""
    raise NotImplementedError(f"ここを実装してください。ヒント: {hint}")

def self_correcting_loop(task: str, max_attempts: int = 3, verbose: bool = True):
    previous_error = None
    for attempt in range(1, max_attempts + 1):
        prompt = build_prompt(task, previous_error)
        if verbose:
            print(f"=== 試行 {attempt} ===")
            print("🧑 モデルへの指示:")
            print(prompt)
            print()

        raw = generate_code(prompt)
        code_str = extract_code(raw)
        if verbose:
            print("🤖 モデルの回答(コード):")
            print(code_str)
            print()

        is_valid, error = check_code(code_str)
        if verbose:
            print("🔧 チェッカーの判定:", "✅ 合格" if is_valid else f"❌ {error}")
            print()

        # TODO: is_valid が True なら code_str を返して終了。
        # False なら、previous_error に error を代入して次のループへ
        TODO("if is_valid: return code_str / else: previous_error = error")
    print("規定回数までに、構文的に正しいコードが得られませんでした。")
    return None


In [ ]:
# --- レスキューセル ---
def self_correcting_loop(task: str, max_attempts: int = 3, verbose: bool = True):
    previous_error = None
    for attempt in range(1, max_attempts + 1):
        prompt = build_prompt(task, previous_error)
        if verbose:
            print(f"=== 試行 {attempt} ===")
            print("🧑 モデルへの指示:")
            print(prompt)
            print()

        raw = generate_code(prompt)
        code_str = extract_code(raw)
        if verbose:
            print("🤖 モデルの回答(コード):")
            print(code_str)
            print()

        is_valid, error = check_code(code_str)
        if verbose:
            print("🔧 チェッカーの判定:", "✅ 合格" if is_valid else f"❌ {error}")
            print()

        if is_valid:
            return code_str
        else:
            previous_error = error
    print("規定回数までに、構文的に正しいコードが得られませんでした。")
    return None

result = self_correcting_loop("2で割り切れるかどうかを判定する関数")
print()
print("最終的なコード:")
print(result)

## 実際に動かしてみて、確認しましょう

上のセルの出力を確認してください。**🧑→🤖→🔧の順番で、1つのやり取りが
繰り返されている**のが見えるはずです。1回目はわざとコロンが抜けたコードで
失敗し、そのエラーメッセージが2回目の🧑の指示文に含まれているのを確認して
ください。

> 💭 **考えてみましょう:** このループが「賢い」ように見えるのは、
> ステップが増えたからでしょうか、それとも**エラーメッセージを次の指示文に
> 使っている**からでしょうか? 試しに`build_prompt`を書き換えて、
> `previous_error`を指示文に含めないようにしたら、モデルは直せると
> 思いますか?

## 自分でタスクを試せるループ

コードを書き換えずに、いろいろなタスクを試せるように、簡単な入力ループを
作りましょう。ステップ4とまったく同じ`self_correcting_loop`の呼び方を、
繰り返すだけです。

使い方: 下のセルを実行すると、タスクを聞かれます。日本語で入力してEnterを
押してください。`終了`と入力するとループを抜けます。

In [ ]:
def interactive_self_correct():
    print("作ってほしい関数を、日本語で説明してください(終了するには「終了」と入力してEnter)")
    while True:
        task = input("タスク: ")
        if task.strip() in ("終了", "quit", "exit", ""):
            print("終了します。")
            break
        self_correcting_loop(task)
        print()

interactive_self_correct()

## 発展: 自分の手で、SQLにも対応させる

ここまでのループは、Pythonのコードしかチェックできません。**このノートブックを
自分の手で拡張して、SQLにも対応させてみましょう。** これは、モジュール2・
拡張演習で「自分の手でMCPツールを追加した」のと同じ考え方です——今回は、
新しいツール(SQLの構文チェッカー)を、このループに追加します。

**使うライブラリ:** `sqlglot`という、純粋にPythonだけで書かれたSQLパーサーを
使います。Node.jsのような別のランタイムのインストールは不要です(`ast`が
Python標準ライブラリの一部であるのと同じように、`sqlglot`は`%pip install`
だけで完結します)。

In [ ]:
%pip install --quiet sqlglot

### あなたの番: `check_sql_code`を書く

`check_code`(Python版)と同じパターンで、SQL版を自分の手で書いてください。
`sqlglot.parse_one(sql_str)`は、SQLが正しければ何も起きず、間違っていれば
`sqlglot.errors.ParseError`という例外を発生させます。

In [ ]:
def check_sql_code(sql_str: str):
    """SQL文字列が構文的に正しいかを調べる。
    正しければ (True, None)、間違っていれば (False, エラーメッセージ) を返す。"""
    import sqlglot
    # TODO: sqlglot.parse_one(sql_str) を試し、
    # sqlglot.errors.ParseError が発生したら (False, エラーメッセージ) を、
    # 発生しなければ (True, None) を返してください
    TODO("try: sqlglot.parse_one(sql_str) を試し、except sqlglot.errors.ParseError as e: で捕まえる")


In [ ]:
# --- レスキューセル ---
def check_sql_code(sql_str: str):
    """SQL文字列が構文的に正しいかを調べる。
    正しければ (True, None)、間違っていれば (False, エラーメッセージ) を返す。"""
    import sqlglot
    try:
        sqlglot.parse_one(sql_str)
        return True, None
    except sqlglot.errors.ParseError as e:
        return False, str(e).splitlines()[0]

# 動作確認
print(check_sql_code("SELECT id, stock FROM inventory WHERE stock < 5"))  # 正しいSQL
print(check_sql_code("SELECT id stock FROM WHERE stock < 5"))              # わざと壊したSQL

### ループをSQLにも対応させる(用意済みのコード)

`self_correcting_loop`が、指定したチェッカー関数(`check_code`または
`check_sql_code`)を使えるように、少しだけ書き換えます。ここは用意済みの
コードです——皆さんが今書いた`check_sql_code`を、既存のループに
差し込むだけです。

In [ ]:
def self_correcting_loop_v2(task: str, checker, language: str = "python", max_attempts: int = 3, verbose: bool = True):
    previous_error = None
    for attempt in range(1, max_attempts + 1):
        if language == "python":
            prompt = build_prompt(task, previous_error)
        else:
            if previous_error is None:
                prompt = f"Write a single SQL query that does the following: {task}\nReturn ONLY a sql code block."
            else:
                prompt = (
                    f"Your previous SQL for this task failed with this error:\n{previous_error}\n"
                    f"Task: {task}\nFix the SQL. Return ONLY a corrected sql code block."
                )

        if verbose:
            print(f"=== 試行 {attempt} ===")
            print("🧑 モデルへの指示:")
            print(prompt)
            print()

        raw = generate_code(prompt)
        code_str = extract_code(raw)
        if verbose:
            print("🤖 モデルの回答:")
            print(code_str)
            print()

        is_valid, error = checker(code_str)
        if verbose:
            print("🔧 チェッカーの判定:", "✅ 合格" if is_valid else f"❌ {error}")
            print()

        if is_valid:
            return code_str
        else:
            previous_error = error
    print("規定回数までに、正しいコードが得られませんでした。")
    return None

In [ ]:
# mock_generate_code はPython用に作られているため、SQLタスクでは常に
# 正しいコードを返すよう、SQL版の模擬関数も簡単に用意しておきます
def mock_generate_code_sql(prompt: str) -> str:
    return "```sql\nSELECT id, stock FROM inventory WHERE stock < 5\n```"

if USE_MOCK_LLM:
    generate_code = mock_generate_code_sql

result = self_correcting_loop_v2(
    "在庫が5未満の商品のIDと在庫数を取得する",
    checker=check_sql_code,
    language="sql",
)
print()
print("最終的なSQL:")
print(result)

# 元に戻す(このセルの後、Pythonのタスクにまた戻れるように)
generate_code = mock_generate_code if USE_MOCK_LLM else real_generate_code

> 💭 **確認できましたか?** ステップ1〜4で作ったループの仕組み(生成→検査→
> 修正)は、そのままSQLにも使い回せました。変わったのは、**「ツール」
> (チェッカー)を差し替えただけ**です。これは、モジュール2で学んだ
> 「エージェントのしくみは同じまま、ツールだけを追加・交換できる」という
> 考え方と、まったく同じです。

## ボーナス(任意): 本物のモデルで試す

`USE_MOCK_LLM = False`に切り替えて、セットアップセルから再実行してみましょう。
本物の`Qwen2.5-0.5B-Instruct`(0.5Bという小さなモデル)が、本当に自分の
構文エラーを直せるか、`verbose=True`のまま観察してみてください。

> ⚠️ **正直な注意:** モジュール3のボーナスで確認した通り、この小さなモデルは
> 不安定な挙動を見せることがあります。コードを書くタスクでも、同じように
> 1回目で失敗したり、`max_attempts`まで直せなかったりする可能性があります。
> **これは想定内です。** 「小さなモデルが、どこまで自己修正できるか」を
> 実際に観察すること自体が、この演習の目的です。同じタスクを複数回試して、
> 毎回同じ結果になるかどうかも確認してみてください。

## まとめ

モジュール3で作ったのは、1回だけ判断して終わるエージェントでした。今日は、
**ツールからのフィードバックを使って、自分の失敗を直す**という、もう一段階
賢いパターンを体験しました。`verbose=True`で見えた🧑→🤖→🔧のやり取りは、
実際のコーディングエージェント(GitHub Copilot、Claude Codeなど)が
使っている考え方の、簡略版です。